# LRao batch-size × cutoff ablation (IID multi)

Batch size is a hidden LRao-specific regularizer: the LFI loss inverts a
**per-batch** covariance of the score, so the batch sets the quality of the
Σ̂ the objective sees. This notebook sweeps it against the cutoff.

**Grid.** cutoff ∈ [1e-3, 1e-5, none]; n ∈ [128, 256, 1024, 2048] training
samples; batch ∈ [256, 512, 1024, 2048] where it differentiates (settings
≥ n collapse to full-batch and are trained once). **No train/val split:**
the model trains on ALL n samples; validation is a FRESH draw of 10%·n
pixels from the pool, disjoint from the training set (drawn right after the
training prefix). Seeds default 42–44 (extend later — resume-safe). 1000
epochs, no wd/clip, robust IQR front-end, [128] ReLU, published
pools/planting, Pd@Pfa=0.1.

**Per run we record three models/epochs:**
- **best-val** — global argmin of the validation LFI cost (10% split, checked
  every epoch); checkpoint saved. The deployable selection rule.
- **best-detection** — argmax test Pd over per-50-epoch snapshots (oracle
  diagnostic, metrics only). The question: does best-val predict it?
- **final** — epoch 1000; checkpoint saved.

**Budget:** 27 runs/seed (3+3+9+12); epoch cost scales with n (not batch).
≈ **2–2.5 h per seed on a T4** (~7 h for 3 seeds; ~2 h on an A100).
Resume-safe: interrupt and rerun the sweep cell; analysis works on partials.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, json, time, copy, contextlib
import numpy as np
import torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
# ----------------- knobs -----------------
N_LIST   = [128, 256, 1024, 2048]
BATCHES  = [256, 512, 1024, 2048]
CUTOFFS  = [1e-3, 1e-5, 0.0]           # 0.0 = no regularization
# batch only differentiates when bs < n_fit = n - val; smaller n collapse to
# full-batch, so they are trained ONCE at their effective (full) batch:
SEEDS    = [42, 43, 44]                # extend to [42..46] later (resume-safe)
MAX_EPOCHS = 1000
SNAP = 50
VAL_FRACTION = 0.1

def batches_for(n):
    # the model trains on ALL n samples (val is a separate fresh draw)
    return sorted({min(b, n) for b in BATCHES})
OUT = 'results/lrao_batch'

def clab(c):
    return 'none' if c == 0.0 else f'{c:.0e}'

# published multi vs-n lines, log-interpolated to this n grid (approximate)
_PN = [20, 40, 60, 100, 200, 500, 1000, 2000]
_PD = [0.287, 0.300, 0.327, 0.390, 0.402, 0.447, 0.537, 0.595]
_PL = [0.329, 0.367, 0.465, 0.501, 0.509, 0.565, 0.561, 0.553]
REF = {n: {'DART': float(np.interp(np.log(n), np.log(_PN), _PD)),
           'LRao': float(np.interp(np.log(n), np.log(_PN), _PL))}
       for n in N_LIST}

In [ ]:
# ----------------- protocol + self-contained LRao math -----------------
import yaml
from tqdm.auto import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import plant_targets
from repro.core.models import ScoreNet
from repro.core.normalization import robust_whitening_iqr

cfg = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
cfg.update(dataset='repro/data/pavia-u.mat')


def build_data(seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    data, gt = load_hsi(cfg['dataset'])
    bkg, tgt = build_pools(data, gt.flatten(), cfg, 'multi')
    s = tgt.mean(axis=0).astype(np.float32)
    idx = np.arange(len(bkg)); rng.shuffle(idx)
    shuf = bkg[idx]
    need = max(N_LIST) + max(1, int(max(N_LIST) * VAL_FRACTION))
    assert len(shuf) >= need + int(cfg['test_size'])
    pool = shuf[:need].astype(np.float32)
    te = shuf[-int(cfg['test_size']):].astype(np.float32)
    planted, labels, _ = plant_targets(te, s, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    return pool, planted.astype(np.float32), labels, s


def lfi_loss(model, batch, cutoff, detach_sigma=True):
    n = len(batch)
    ctx = torch.no_grad() if detach_sigma else contextlib.nullcontext()
    with ctx:
        psi0 = model(batch)
        mu = psi0.mean(0)
        c = psi0 - mu
        Sigma = (c.T @ c) / max(n - 1, 1)
        U, S, Vh = torch.linalg.svd(Sigma)
        thr = float(cutoff) * S[0]
        S_inv = torch.where(S > thr, 1.0 / S, torch.zeros_like(S))
        Sigma_inv = Vh.T @ torch.diag(S_inv) @ U.T
    from torch.func import jacrev, vmap
    J = vmap(jacrev(lambda x: model(x.unsqueeze(0)).squeeze(0)))(batch)
    G = J.mean(0)
    return -(G.T @ Sigma_inv @ G).trace()


@torch.no_grad()
def lrao_score(model, train_data, test_data, s, cutoff, delta=0.01):
    model.eval()
    d = train_data.shape[1]
    X_tr = torch.tensor(train_data, dtype=torch.float32, device=DEVICE)
    X_te = torch.tensor(test_data, dtype=torch.float32, device=DEVICE)
    I_d = torch.eye(d, device=DEVICE)
    psi_tr = model(X_tr).cpu().numpy().astype(np.float64)
    if not np.all(np.isfinite(psi_tr)):
        return np.zeros(len(test_data))
    mu = psi_tr.mean(0)
    Sigma = (psi_tr - mu).T @ (psi_tr - mu) / max(len(train_data) - 1, 1)
    U, S, Vh = np.linalg.svd((Sigma + Sigma.T) / 2)
    thr = float(cutoff) * S[0]
    S_inv = np.where(S > thr, 1.0 / S, 0.0)
    Sigma_inv = Vh.T @ np.diag(S_inv) @ U.T
    G = np.zeros((psi_tr.shape[1], d))
    for j in range(d):
        plus = model(X_tr + delta * I_d[j]).cpu().numpy()
        minus = model(X_tr - delta * I_d[j]).cpu().numpy()
        G[:, j] = ((plus - minus) / (2.0 * delta)).mean(0)
    g_s = G @ np.asarray(s, np.float64)
    J_s = float(g_s @ Sigma_inv @ g_s)
    psi_te = model(X_te).cpu().numpy()
    return (psi_te - mu) @ (Sigma_inv @ g_s) / np.sqrt(max(J_s, 1e-12))


def metrics(labels, sc):
    return dict(pd=_pd_at_fa(labels, sc, cfg['pfa']), auc=_auc(labels, sc))

In [ ]:
# ----------------- trainer: best-val + best-detection + final -----------------
def train_one(tr, val, cutoff, bs_req, seed, label, planted, labels, s,
              ckpt_dir):
    """tr = ALL n training samples; val = a fresh disjoint 10% draw."""
    torch.manual_seed(seed)
    W = robust_whitening_iqr(tr)
    net = ScoreNet(tr.shape[1], [128], 'relu', whitening=W).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'], weight_decay=0.0)
    Xf = torch.tensor(tr).to(DEVICE)
    Xv = torch.tensor(val).to(DEVICE)
    N = len(Xf); bs = min(int(bs_req), N)
    best_val, bv_state, bv_epoch = float('inf'), None, 0
    val_hist, snaps = [], []
    pbar = tqdm(range(1, MAX_EPOCHS + 1), desc=label, leave=False)
    for ep in pbar:
        net.train()
        perm = torch.randperm(N)
        for i in range(0, N, bs):
            try:
                loss = lfi_loss(net, Xf[perm[i:i + bs]], cutoff)
            except Exception:
                continue
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        try:
            vl = float(lfi_loss(net, Xv, cutoff).detach())
        except Exception:
            vl = float('nan')
        val_hist.append(vl)
        if np.isfinite(vl) and vl < best_val:
            best_val, bv_epoch = vl, ep
            bv_state = copy.deepcopy(net.state_dict())
        if ep % SNAP == 0:
            sc = lrao_score(net, tr, planted, s, cutoff)
            snaps.append({'epoch': ep, **metrics(labels, sc)})
            pbar.set_postfix(pd=f"{snaps[-1]['pd']:.3f}", bv=bv_epoch)
    final_state = copy.deepcopy(net.state_dict())
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({'state_dict': {k: v.cpu() for k, v in bv_state.items()},
                'epoch': bv_epoch},
               os.path.join(ckpt_dir, label + '_bestval.pt'))
    torch.save({'state_dict': {k: v.cpu() for k, v in final_state.items()},
                'epoch': MAX_EPOCHS},
               os.path.join(ckpt_dir, label + '_final.pt'))
    best_snap = max(snaps, key=lambda t: t['pd'])
    out = {'bv_epoch': bv_epoch, 'bd_epoch': best_snap['epoch'],
           'bd': {'pd': best_snap['pd'], 'auc': best_snap['auc']},
           'val_hist': [round(v, 6) for v in val_hist], 'snaps': snaps}
    for kind, state in (('bv', bv_state), ('final', final_state)):
        net.load_state_dict(state); net.eval()
        sc = lrao_score(net, tr, planted, s, cutoff)
        out[kind if kind == 'final' else 'bv_model'] = metrics(labels, sc)
    return out

In [ ]:
# ----------------- the sweep (resume-safe) -----------------
os.makedirs(OUT, exist_ok=True)
met_path = os.path.join(OUT, 'metrics.json')
rec = json.load(open(met_path)) if os.path.exists(met_path) else {}
rec['_meta'] = dict(n_list=N_LIST, batches=BATCHES, seeds=SEEDS,
                    cutoffs=[clab(c) for c in CUTOFFS],
                    max_epochs=MAX_EPOCHS, val_fraction=VAL_FRACTION)
t0 = time.time()
for seed in SEEDS:
    pool, planted, labels, s = build_data(seed)
    for n in N_LIST:
        tr = pool[:n]
        val = pool[n:n + max(1, int(n * VAL_FRACTION))]   # fresh, disjoint
        for bs in batches_for(n):
            for co in CUTOFFS:
                key = f'c{clab(co)}_b{bs}_n{n}_s{seed}'
                if rec.get(key, {}).get('final'):
                    continue
                t1 = time.time()
                r = train_one(tr, val, co, bs, seed, key, planted, labels, s,
                              os.path.join(OUT, 'ckpt'))
                r['sec'] = round(time.time() - t1)
                rec[key] = r
                json.dump(rec, open(met_path, 'w'))
                print(f"{key}: bv ep{r['bv_epoch']} Pd={r['bv_model']['pd']:.3f}"
                      f" | bd ep{r['bd_epoch']} Pd={r['bd']['pd']:.3f}"
                      f" | final Pd={r['final']['pd']:.3f} ({r['sec']}s)",
                      flush=True)
print(f'TOTAL {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# ----------------- analysis + figures (partial-safe) -----------------
import matplotlib.pyplot as plt
from IPython.display import Image, display

FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG, exist_ok=True)
rec = json.load(open(met_path))
CL = [clab(c) for c in CUTOFFS]

def get(cl, bs, n, sd):
    return rec.get(f'c{cl}_b{bs}_n{n}_s{sd}')

def show(fig, name):
    fig.tight_layout()
    p = os.path.join(FIG, name + '.png')
    fig.savefig(p, dpi=200); fig.savefig(p.replace('.png', '.pdf'))
    plt.close(fig); display(Image(p, width=980))

# --- fig 1: Pd (best-val model) vs batch, panel per n, lines per cutoff ---
fig, axes = plt.subplots(1, len(N_LIST), figsize=(4.0 * len(N_LIST), 4.0),
                         sharey=True, squeeze=False)
cols = plt.cm.viridis(np.linspace(0, 0.8, len(CL)))
for a, n in zip(axes[0], N_LIST):
    B = batches_for(n)
    for i, cl in enumerate(CL):
        m = [np.nanmean([r['bv_model']['pd'] for sd in SEEDS
                         if (r := get(cl, bs, n, sd))] or [np.nan])
             for bs in B]
        a.plot(B, m, 'o-', color=cols[i], lw=1.6, label=f'cutoff {cl}')
        mf = [np.nanmean([r['final']['pd'] for sd in SEEDS
                          if (r := get(cl, bs, n, sd))] or [np.nan])
              for bs in B]
        a.plot(B, mf, 'o:', color=cols[i], lw=1, alpha=0.6)
    if n in REF:
        a.axhline(REF[n]['LRao'], color='k', ls='--', lw=1, label='LRao (pub, interp)')
        a.axhline(REF[n]['DART'], color='tab:red', ls=':', lw=1,
                  label='DART (pub, interp)')
    a.set_xscale('log', base=2); a.set_xticks(B)
    a.set_xticklabels(B); a.minorticks_off()
    a.set_xlabel('effective batch size')
    a.set_title(f'n={n}' + (' (full-batch only)' if len(B) == 1 else ''))
    a.grid(alpha=0.3)
axes[0][0].set_ylabel('Pd@0.1 (solid=best-val, dotted=final)')
axes[0][-1].legend(fontsize=7)
fig.suptitle('LRao: batch size x cutoff (mean over seeds)')
show(fig, 'batch_cutoff_pd')

# --- fig 2: best-val epoch vs best-detection epoch (correlation) ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
all_bs = sorted({b for n in N_LIST for b in batches_for(n)})
_mk = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
mark = {b: _mk[i % len(_mk)] for i, b in enumerate(all_bs)}
xs, ys = [], []
for i, cl in enumerate(CL):
    for n in N_LIST:
        for bs in batches_for(n):
            for sd in SEEDS:
                r = get(cl, bs, n, sd)
                if not r:
                    continue
                axes[0].scatter(r['bv_epoch'], r['bd_epoch'], s=22,
                                color=cols[i], marker=mark[bs], alpha=0.6)
                xs.append(r['bv_epoch']); ys.append(r['bd_epoch'])
                axes[1].scatter(r['bv_model']['pd'], r['bd']['pd'], s=22,
                                color=cols[i], marker=mark[bs], alpha=0.6)
lim = [0, MAX_EPOCHS]
axes[0].plot(lim, lim, 'k--', lw=1)
if len(xs) > 2:
    axes[0].set_title('epochs: best-val vs best-detection '
                      f'(r={np.corrcoef(xs, ys)[0, 1]:.2f})')
axes[0].set_xlabel('best-VAL epoch'); axes[0].set_ylabel('best-DETECTION epoch')
axes[0].grid(alpha=0.3)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('Pd of best-val model')
axes[1].set_ylabel('Pd at best-detection snapshot (oracle)')
axes[1].set_title('what val selection costs vs the oracle')
axes[1].grid(alpha=0.3)
for i, cl in enumerate(CL):
    axes[0].scatter([], [], color=cols[i], label=f'cutoff {cl}')
for bs in all_bs:
    axes[0].scatter([], [], color='gray', marker=mark[bs], label=f'b={bs}')
axes[0].legend(fontsize=6.5, ncol=2)
show(fig, 'val_vs_detection_correlation')

# --- summary ---
lines = ['# LRao batch x cutoff — Pd@0.1 (best-val model, mean over seeds)', '']
for n in N_LIST:
    B = batches_for(n)
    lines += [f'## n={n}',
              '| cutoff | ' + ' | '.join(f'b={b}' for b in B) + ' |',
              '|' + '---|' * (len(B) + 1)]
    for cl in CL:
        vals = [np.nanmean([r['bv_model']['pd'] for sd in SEEDS
                            if (r := get(cl, bs, n, sd))] or [np.nan])
                for bs in B]
        lines.append(f'| {cl} | ' + ' | '.join(
            f'{v:.3f}' if np.isfinite(v) else '—' for v in vals) + ' |')
    lines.append('')
open(os.path.join(OUT, 'summary.md'), 'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ----------------- display all saved figures -----------------
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('results/lrao_batch/figures/*.png')):
    print(p)
    display(Image(filename=p, width=980))

In [ ]:
# ----------------- zip -----------------
!zip -qr lrao_batch_light.zip results/lrao_batch -x "*/ckpt/*"
!zip -qr lrao_batch_full.zip results/lrao_batch
!ls -lh lrao_batch_*.zip
try:
    from google.colab import files
    files.download('lrao_batch_light.zip')
except Exception as e:
    print('manual download:', e)